# 寶石方塊：8×8 辨色與自動交換
以原本 `寶石.ipynb` 的 PyAutoGUI 座標與三連線判斷為基礎。這是操作既有遊戲的助手，不是另一個遊戲。

1. 執行最後一格後等待 5 秒，請切換至遊戲，確保棋盤完整可見且已靜止。
2. 左上寶石中心為 `(1370, 366)`；高速模式一次截取棋盤並讀取 64 格，不再移動滑鼠逐格取色。
3. 參考提供圖片，水平與垂直中心間距均設為 50 像素；若實際遊戲縮放不同，請修改下方 gap_x / gap_y。
4. 倒數後計時 85 秒：一次讀取 64 格 → 判斷三連線交換 → 拖曳一次 → 等待畫面穩定 → 重新讀取。連續四張畫面檢查，允許最多四個特殊、未知或閃爍格不參與連線，其餘穩定格繼續操作；大量變動則等待。取色與動畫等待也計入 85 秒。
5. 高分策略加權四連、五連與交叉連線；連鎖最大八層，布局最多預看後續三手、每層最多三個候選。額外搜尋採 0.15 毫秒軟預算，時間不足立即使用已有結果，未必每次能達到最大深度，縮短預算也可能降低實際深度。0.5 ms 目標僅指找交換與排序，不含截圖、存檔、滑鼠與動畫；Python 及作業系統排程無法保證每次硬性低於 0.5 ms，也不保證分數提高。
6. 優先選擇立即消除內含黑色標記寶石的合法交換，再比較一般策略值；標記不依賴寶石底色。只檢查寶石內部並排除大面積黑色，仍可能誤認陰影，可調整 BLACK_THRESHOLD / BLACK_MIN_PIXELS / BLACK_MAX_RATIO。黑色偵測屬影像處理，不包含在純選步的 0.5 ms 目標內。
7. 自動操作時將滑鼠移到螢幕左上角可緊急停止，也能中斷核心。

執行全部儲存格時，第一個程式碼儲存格會檢查套件，缺少時自動安裝到目前 Notebook 核心；首次安裝需要網路。Notebook 請選 `iem_python` 環境核心。支援一般相鄰交換三消；特殊寶石、障礙與技能尚未建模。

In [ ]:
# 安裝到目前 Notebook 核心，不使用可能指向其他環境的 pip。
import importlib
import importlib.util
import subprocess
import sys

print('目前 Python：', sys.executable)
missing_packages = [package for module, package in
                    [('pyautogui', 'pyautogui'), ('PIL', 'pillow')]
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    print('正在安裝：', ', '.join(missing_packages), flush=True)
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('套件安裝失敗，請提供此儲存格的完整安裝輸出。尚未啟動滑鼠操作。') from exc
    importlib.invalidate_caches()
# 實際匯入確認安裝成功；後面的程式碼才開始載入遊戲助手。
import pyautogui
from PIL import Image
print('PyAutoGUI 與 Pillow 已可使用。')


In [ ]:
import colorsys
import json
import math
import time
from pathlib import Path
from statistics import median
import pyautogui as pyg
from PIL import ImageStat, ImageChops

pyg.FAILSAFE = True
pyg.PAUSE = 0.0  # 移除每次 PyAutoGUI 呼叫的額外等待，仍保留緊急停止。
SIZE = 8
# 起點是第一顆寶石中心，不是棋盤外框。
basex, basey = 1370, 366
# 最新桌面截圖：左上中心約 (1370, 366)，右下中心約 (1720, 716)。
# 兩端各跨 7 格，水平與垂直格距均為 350 / 7 = 50 px。
gap_x = gap_y = 50.0
PALETTE = {}  # 留空使用七色 HSV 辨識；可選擇手動校正覆寫。
SAMPLE_RADIUS = 3
COLOR_TOLERANCE = 65.0
COLOR_MARGIN = 12.0
STABLE_FRAMES = 4
MAX_UNCERTAIN_CELLS = 4  # 少量閃爍或未知格不再阻塞整盤。
STABLE_RGB_DELTA = 12.0
BLACK_THRESHOLD = 45  # 三個 RGB 通道皆低於此值，才算近黑色。
BLACK_MIN_PIXELS = 8
BLACK_MAX_RATIO = 0.35  # 排除大面積暗背景；只觀察寶石中心內部。

def gem_position(r, c):
    return round(basex + c * gap_x), round(basey + r * gap_y)

def countdown(message, seconds=5):
    print(message, flush=True)
    for remaining in range(seconds, 0, -1):
        print(remaining, end=' ', flush=True)
        pyg.sleep(1)
    print()
    pyg.failSafeCheck()

def sample_rgb(image, x, y, radius=SAMPLE_RADIUS):
    # 區域中位數比單一中心像素不易受反光或邊緣影響。
    if not (radius <= x < image.width-radius and radius <= y < image.height-radius):
        raise ValueError('取樣位置超出截圖範圍，請重新校正。')
    # Pillow 原生直方圖計算中位數，避免 Python 逐像素讀取與排序。
    patch = image.crop((x-radius, y-radius, x+radius+1, y+radius+1))
    return tuple(ImageStat.Stat(patch).median[:3])

def rgb_distance(a, b):
    return math.sqrt(sum((x-y)**2 for x, y in zip(a, b)))

def classify(rgb):
    if PALETTE:
        distances = sorted((rgb_distance(rgb, ref), name) for name, ref in PALETTE.items())
        if distances[0][0] > COLOR_TOLERANCE:
            return '?'
        if len(distances) > 1 and distances[1][0] - distances[0][0] < COLOR_MARGIN:
            return '?'
        return distances[0][1]
    # 依圖片的七色寶石，用色相區分，降低明暗對 RGB 的影響。
    h, s, v = colorsys.rgb_to_hsv(*(channel / 255 for channel in rgb))
    h *= 360
    if v < 0.30:
        return '?'  # 黑色背景、陰影不當作寶石
    if s < 0.22:
        return '白' if v >= 0.48 else '?'
    if s < 0.38:
        return '?'  # 可能是反光或倒數遮罩，避免猜測
    if h < 15 or h >= 345:
        return '紅'
    if h < 43:
        return '橙'
    if h < 75:
        return '黃'
    if h < 165:
        return '綠'
    if h < 255:
        return '藍'
    return '紫'


In [ ]:
def calibrate_board():
    global basex, basey, gap_x, gap_y
    input('準備好後按 Enter，接著有 5 秒將滑鼠放到左上第一顆寶石中心。')
    countdown('請定位左上角寶石中心')
    x1, y1 = pyg.position()
    input('回到這裡按 Enter，接著有 5 秒定位右下最後一顆寶石中心。')
    countdown('請定位右下角寶石中心')
    x2, y2 = pyg.position()
    if x2 <= x1 or y2 <= y1:
        raise ValueError('右下角必須位於左上角右下方。')
    basex, basey = x1, y1
    gap_x, gap_y = (x2-x1)/7, (y2-y1)/7
    if min(gap_x, gap_y) < 12:
        raise ValueError('格距過小，請確認選的是整個 8×8 棋盤。')
    image = pyg.screenshot().convert('RGB')
    sample_rgb(image, x1, y1)
    sample_rgb(image, x2, y2)
    print(f'左上中心={gem_position(0, 0)}，右下中心={gem_position(7, 7)}，格距={gap_x:.1f}×{gap_y:.1f}')

def calibrate_colors():
    global PALETTE
    palette = {}
    print('輸入每一種普通寶石的顏色名稱，例如 紅、藍、綠、黃、紫、白。')
    print('取樣時請選沒有閃光的普通寶石中心。所有顏色完成後輸入空白。')
    while True:
        name = input('顏色名稱（Enter 完成）：').strip()
        if not name:
            break
        if name == '?':
            print('問號保留給未知顏色，請換名稱。')
            continue
        countdown(f'將滑鼠放到「{name}」寶石中心')
        x, y = pyg.position()
        rgb = sample_rgb(pyg.screenshot().convert('RGB'), x, y)
        palette[name] = rgb
        print(name, rgb)
    if len(palette) < 3:
        raise ValueError('至少需要三種顏色樣本；請重新執行顏色校正。')
    PALETTE = palette

def park_mouse():
    # 移出棋盤，避免滑鼠懸停亮光改變辨色；不點擊任何 UI。
    x = max(1, basex - round(gap_x))
    y = max(1, basey - round(gap_y))
    pyg.moveTo(x, y, duration=0)


In [ ]:
class GemBoard(list):
    def __init__(self, rows, black_cells=()):
        super().__init__(rows)
        self.black_cells = frozenset(black_cells)

def has_inner_black(image, x, y):
    radius = max(4, round(min(gap_x, gap_y)*0.20))
    if x-radius < 0 or y-radius < 0 or x+radius >= image.width or y+radius >= image.height:
        return False
    patch = image.crop((x-radius, y-radius, x+radius+1, y+radius+1))
    red, green, blue = patch.split()
    maximum = ImageChops.lighter(ImageChops.lighter(red, green), blue)
    count = sum(maximum.histogram()[:BLACK_THRESHOLD])
    return BLACK_MIN_PIXELS <= count <= patch.width*patch.height*BLACK_MAX_RATIO

def surrounding_gem_rgb(image, x, y, fallback):
    # 黑字可能覆蓋正中心，改取附近八個小區塊，使用最常見的可辨色樣本。
    distance = max(3, round(min(gap_x, gap_y)*0.22))
    groups = {}
    for dx, dy in ((-1,-1),(0,-1),(1,-1),(-1,0),(1,0),(-1,1),(0,1),(1,1)):
        px, py = x+dx*distance, y+dy*distance
        if not (2 <= px < image.width-2 and 2 <= py < image.height-2):
            continue
        rgb = sample_rgb(image, px, py, radius=2)
        color = classify(rgb)
        if color != '?':
            groups.setdefault(color, []).append(rgb)
    if not groups:
        return fallback
    samples = max(groups.values(), key=len)
    return tuple(round(median(rgb[i] for rgb in samples)) for i in range(3))

def read_board(image=None):
    offset_x = offset_y = 0
    if image is None:
        x1, y1 = gem_position(0, 0)
        x2, y2 = gem_position(7, 7)
        margin = max(SAMPLE_RADIUS, round(min(gap_x, gap_y)*0.32))
        offset_x, offset_y = x1-margin, y1-margin
        width, height = x2-x1+2*margin+1, y2-y1+2*margin+1
        screen_w, screen_h = pyg.size()
        if offset_x < 0 or offset_y < 0 or offset_x+width > screen_w or offset_y+height > screen_h:
            raise ValueError('棋盤超出螢幕範圍，請檢查起點與格距。')
        image = pyg.screenshot(region=(offset_x, offset_y, width, height))
    image = image.convert('RGB')
    rgbs = [[sample_rgb(image, gem_position(r, c)[0]-offset_x,
                        gem_position(r, c)[1]-offset_y) for c in range(SIZE)]
            for r in range(SIZE)]
    black_cells = set()
    for r in range(SIZE):
        for c in range(SIZE):
            x, y = gem_position(r, c)
            x, y = x-offset_x, y-offset_y
            if has_inner_black(image, x, y):
                black_cells.add((r, c))
                rgbs[r][c] = surrounding_gem_rgb(image, x, y, rgbs[r][c])
    board = GemBoard([[classify(rgb) for rgb in row] for row in rgbs], black_cells)
    return board, rgbs

def scan_board_with_mouse(deadline=None):
    # 保留舊函式名稱以相容；高速版不再逐格移動滑鼠。
    pyg.failSafeCheck()
    if deadline is not None and time.monotonic() >= deadline:
        raise TimeoutError('操作時間已到，停止取色。')
    return read_board()

def stable_cells_from_frames(frames):
    # 閃爍、變色或無法辨認的格子僅標成 ?，不把特殊方塊猜成普通色。
    if len(frames) < STABLE_FRAMES:
        return None
    board, rgbs = frames[-1]
    black_cells = set(getattr(board, 'black_cells', ()))
    for old_board, _ in frames[:-1]:
        black_cells.intersection_update(getattr(old_board, 'black_cells', ()))
    safe = GemBoard([row[:] for row in board])
    uncertain = 0
    for r in range(SIZE):
        for c in range(SIZE):
            color = board[r][c]
            if color in ('?', None) or any(
                old_board[r][c] != color or
                rgb_distance(old_rgb[r][c], rgbs[r][c]) > STABLE_RGB_DELTA
                for old_board, old_rgb in frames[:-1]):
                safe[r][c] = '?'
                uncertain += 1
    safe.black_cells = frozenset(pos for pos in black_cells if safe[pos[0]][pos[1]] != '?')
    return (safe, rgbs) if uncertain <= MAX_UNCERTAIN_CELLS else None

def wait_for_stable_board(timeout=10.0):
    deadline = time.monotonic() + timeout
    frames = []
    while time.monotonic() < deadline:
        pyg.failSafeCheck()
        frames.append(read_board())
        frames = frames[-STABLE_FRAMES:]
        result = stable_cells_from_frames(frames)
        if result is not None:
            return result
        pyg.sleep(min(0.05, max(0, deadline-time.monotonic())))
    raise TimeoutError('大量格子仍在變動或無法辨色，等待動畫結束後重試。')

def print_board(board):
    print('列\欄  ' + '  '.join(str(c+1) for c in range(SIZE)))
    for r, row in enumerate(board, 1):
        print(f'{r:>2}    ' + '  '.join(row))

def record_board(board, rgbs, path='gem_board.json'):
    data = {'origin': [basex, basey], 'gap': [gap_x, gap_y],
            'palette': PALETTE, 'board': board, 'rgb': rgbs,
            'black_cells': sorted(getattr(board, 'black_cells', ())),
            'recorded_at': time.strftime('%Y-%m-%d %H:%M:%S')}
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding='utf-8')


In [ ]:
DIRECTIONS = {'上': (-1, 0), '下': (1, 0), '左': (0, -1), '右': (0, 1)}

def match_at(board, r, c):
    color = board[r][c]
    if color in ('?', None):
        return set()
    result = set()
    for dr, dc in ((0, 1), (1, 0)):
        line = {(r, c)}
        for sign in (-1, 1):
            nr, nc = r + sign*dr, c + sign*dc
            while 0 <= nr < SIZE and 0 <= nc < SIZE and board[nr][nc] == color:
                line.add((nr, nc))
                nr, nc = nr + sign*dr, nc + sign*dc
        if len(line) >= 3:
            result.update(line)
    return result

def match_runs(board):
    # 每條橫列、直行只掃一次，取得三連以上的完整線段。
    runs = []
    for vertical in (False, True):
        for fixed in range(SIZE):
            start = 0
            while start < SIZE:
                pos = (start, fixed) if vertical else (fixed, start)
                color = board[pos[0]][pos[1]]
                end = start + 1
                while end < SIZE:
                    r, c = (end, fixed) if vertical else (fixed, end)
                    if board[r][c] != color:
                        break
                    end += 1
                if color not in (None, '?') and end-start >= 3:
                    runs.append({(i, fixed) if vertical else (fixed, i) for i in range(start, end)})
                start = end
    return runs

def all_matches(board):
    runs = match_runs(board)
    return set().union(*runs) if runs else set()

def pattern_value(runs):
    # 策略權重，不代表遊戲實際計分；偏好能製造特殊寶石的形狀。
    if not runs:
        return 0.0
    matched = set().union(*runs)
    value = 10.0 * len(matched)
    value += sum(60 if len(run) >= 5 else 25 if len(run) == 4 else 0 for run in runs)
    value += 40 * sum(bool(a & b) for i, a in enumerate(runs) for b in runs[i+1:])
    return value

def remove_and_drop(board, matched):
    # 不猜測新寶石：空位用 None，僅模擬已知寶石向下掉落。
    result = [[None] * SIZE for _ in range(SIZE)]
    for c in range(SIZE):
        survivors = [board[r][c] for r in range(SIZE)
                     if (r, c) not in matched and board[r][c] is not None]
        for r, color in enumerate(survivors, SIZE-len(survivors)):
            result[r][c] = color
    return result

def best_followup_value(board, deadline, depth=3):
    # 空位不補入猜測顏色；只評估已知寶石真正能形成的下一手。
    if time.monotonic() >= deadline:
        return 0.0
    replies = find_moves(board, deadline=deadline)
    best = 0.0
    evaluated = []
    for reply in replies:
        if time.monotonic() >= deadline:
            break
        value = pattern_value(reply['runs'])
        best = max(best, value)
        evaluated.append((value, reply))
    # 先完成單層估值，再在剩餘時間內深化前三個候選；任何層都能提早返回。
    if depth > 1:
        evaluated.sort(key=lambda pair: pair[0], reverse=True)
        for value, reply in evaluated[:3]:
            if time.monotonic() >= deadline:
                break
            trial = [row[:] for row in board]
            r, c = reply['start']
            nr, nc = reply['end']
            trial[r][c], trial[nr][nc] = trial[nr][nc], trial[r][c]
            matched = set().union(*reply['runs'])
            settled = False
            for _ in range(8):
                if time.monotonic() >= deadline:
                    break
                trial = remove_and_drop(trial, matched)
                runs = match_runs(trial)
                if not runs:
                    settled = True
                    break
                matched = set().union(*runs)
            if settled and time.monotonic() < deadline:
                future = best_followup_value(trial, deadline, depth=depth-1)
                best = max(best, value + 0.5*future)
    # 缺少的隨機補珠越多，對下一手布局的信心就越低。
    known_ratio = sum(color not in (None, '?') for row in board for color in row) / (SIZE*SIZE)
    return best * known_ratio

def rank_moves(board, moves, budget=0.00015, remaining=85):
    # 特殊格的位置、消除效果未知，不跨過它模擬掉落，避免虛假的高分預測。
    if any(color in ('?', None) for row in board for color in row):
        budget = 0
    deadline = time.monotonic() + max(0, budget)
    ranked = []
    for move in moves:
        trial = [row[:] for row in board]
        r, c = move['start']
        nr, nc = move['end']
        trial[r][c], trial[nr][nc] = trial[nr][nc], trial[r][c]
        runs = move.get('runs')
        if runs is None:
            runs = match_runs(trial)
        matched = set().union(*runs) if runs else set()
        # 相同消除品質時略偏好下方交換，讓上方更多寶石有機會重組。
        priority = pattern_value(runs) + 0.05 * (r+nr)
        # 標記跟著寶石交換，不把交換前的位置誤當成交換後的消除位置。
        black_hits = sum((move['end'] if pos == move['start'] else
                          move['start'] if pos == move['end'] else pos) in matched
                         for pos in getattr(board, 'black_cells', ()))
        ranked.append({**move, 'priority': priority, 'cascade_estimate': 0, 'followup_value': 0.0,
                       'black_hits': black_hits, '_trial': trial, '_matched': matched})
    ranked.sort(key=lambda m: (m['black_hits'], m['priority']), reverse=True)
    # 先保證每一手都有即時評分，再在時間內評估已知寶石的連鎖。
    for i, move in enumerate(ranked):
        trial, matched = move.pop('_trial'), move.pop('_matched')
        for depth in range(1, 9):
            # 留一部分預算評估下一手，避免連鎖搜尋吃光全部時間。
            if time.monotonic() >= deadline or (i >= 3 and deadline-time.monotonic() < budget*0.4):
                break
            trial = remove_and_drop(trial, matched)
            runs = match_runs(trial)
            if not runs:
                move['_settled'] = trial
                break
            matched = set().union(*runs)
            move['priority'] += (0.8 ** depth) * pattern_value(runs)
            move['cascade_estimate'] += 1
    ranked.sort(key=lambda m: (m['black_hits'], m['priority']), reverse=True)
    # 布局需要時間兌現：最後三秒不再為未來一手額外加分。
    future_weight = 0.35 * min(1.0, max(0.0, (remaining-3.0)/7.0))
    for i, move in enumerate(ranked):
        settled = move.pop('_settled', None)
        if i < 6 and settled is not None and future_weight > 0 and time.monotonic() < deadline:
            followup = best_followup_value(settled, deadline)
            move['followup_value'] = followup
            move['priority'] += future_weight * followup
    return sorted(ranked, key=lambda m: (m['black_hits'], m['priority']), reverse=True)

def affected_swap_runs(work, start, end):
    # work 是單次搜尋的私有副本；交換後只檢查兩個端點，最後一定還原。
    r, c = start
    nr, nc = end
    work[r][c], work[nr][nc] = work[nr][nc], work[r][c]
    runs, seen = [], set()
    try:
        for row, col in (start, end):
            color = work[row][col]
            for dr, dc in ((0, 1), (1, 0)):
                before = after = 0
                y, x = row-dr, col-dc
                while 0 <= y < SIZE and 0 <= x < SIZE and work[y][x] == color:
                    before += 1
                    y, x = y-dr, x-dc
                y, x = row+dr, col+dc
                while 0 <= y < SIZE and 0 <= x < SIZE and work[y][x] == color:
                    after += 1
                    y, x = y+dr, x+dc
                if before+after < 2:
                    continue
                # 先用整數計數排除無效線，確定三連後才建立集合。
                key = (row-before*dr, col-before*dc, dr, dc, before+after+1)
                if key not in seen:
                    seen.add(key)
                    runs.append({(row+i*dr, col+i*dc) for i in range(-before, after+1)})
        return runs
    finally:
        work[r][c], work[nr][nc] = work[nr][nc], work[r][c]

def swap_matches(board, start, end):
    r, c = start
    nr, nc = end
    if not all(0 <= v < SIZE for v in (r, c, nr, nc)):
        return set()
    if abs(r-nr) + abs(c-nc) != 1:
        return set()
    if board[r][c] in ('?', None) or board[nr][nc] in ('?', None):
        return set()
    if board[r][c] == board[nr][nc]:
        return set()
    # 使用副本模擬交換，不修改已記錄的棋盤。
    trial = [row[:] for row in board]
    runs = affected_swap_runs(trial, start, end)
    return set().union(*runs) if runs else set()

def can_move(board, r, c, direction):
    dr, dc = DIRECTIONS[direction]
    return bool(swap_matches(board, (r, c), (r+dr, c+dc)))

def can_move_left(r, c, board):
    return can_move(board, r, c, '左')

def can_move_right(r, c, board):
    return can_move(board, r, c, '右')

def can_move_up(r, c, board):
    return can_move(board, r, c, '上')

def can_move_down(r, c, board):
    return can_move(board, r, c, '下')

def find_moves(board, deadline=None):
    moves = []
    work = [row[:] for row in board]
    for r in range(SIZE):
        for c in range(SIZE):
            # 右、下即可枚舉所有相鄰交換，避免重複列出反方向。
            if deadline is not None and time.monotonic() >= deadline:
                return sorted(moves, key=lambda move: move['score'], reverse=True)
            for direction in ('右', '下'):
                dr, dc = DIRECTIONS[direction]
                end = (r+dr, c+dc)
                nr, nc = end
                if nr >= SIZE or nc >= SIZE:
                    continue
                if work[r][c] in ('?', None) or work[nr][nc] in ('?', None) or work[r][c] == work[nr][nc]:
                    continue
                runs = affected_swap_runs(work, (r, c), end)
                if runs:
                    moves.append({'start': (r, c), 'end': end,
                                  'direction': direction, 'score': len(set().union(*runs)), 'runs': runs})
    return sorted(moves, key=lambda move: move['score'], reverse=True)


In [ ]:
def auto_play(seconds=85):
    deadline = time.monotonic() + seconds
    completed = 0
    previous = None
    last_move = None
    rejected = set()
    next_status_at = 0.0
    cached_key, cached_moves = None, []
    def status(message):
        nonlocal next_status_at
        if time.monotonic() >= next_status_at:
            print(message)
            next_status_at = time.monotonic() + 2.0
    try:
        while time.monotonic() < deadline:
            try:
                # 四張截圖逐格確認，少量閃爍特殊格不再阻塞整盤。
                board, rgbs = wait_for_stable_board(timeout=min(3.0, max(0, deadline-time.monotonic())))
            except TimeoutError as exc:
                if time.monotonic() >= deadline:
                    break
                status(f'暫不交換，重新等待棋盤：{exc}')
                continue
            board_key = (tuple(tuple(row) for row in board), tuple(sorted(getattr(board, 'black_cells', ()))))
            if board_key != cached_key:
                record_board(board, rgbs)
                cached_moves = find_moves(board)
                cached_key = board_key
            if time.monotonic() >= deadline:
                break
            if any(color == '?' for row in board for color in row):
                status('特殊或閃爍格暫不參與交換，其餘穩定寶石繼續判斷。')
            if all_matches(board):
                status('棋盤仍有變化或消除尚未結束，重新記錄。')
                pyg.sleep(min(0.1, max(0, deadline-time.monotonic())))
                continue
            if previous == board:
                if last_move is not None:
                    rejected.add(last_move)
                    print('上次交換後棋盤未改變，本棋盤略過該交換。')
            else:
                rejected.clear()
            moves = [m for m in cached_moves if (m['start'], m['end']) not in rejected]
            if not moves:
                status('目前沒有可用交換，等待遊戲變化後重新記錄。')
                pyg.sleep(min(0.1, max(0, deadline-time.monotonic())))
                continue
            # 保留移動、拖曳與釋放時間，期限前不再啟動來不及完成的交換。
            if deadline - time.monotonic() < 0.4:
                break
            moves = rank_moves(board, moves, budget=min(0.00015, max(0, deadline-time.monotonic()-0.4)),
                               remaining=max(0, deadline-time.monotonic()))
            if deadline-time.monotonic() < 0.4:
                break
            move = moves[0]
            r, c = move['start']
            nr, nc = move['end']
            print(f'第 {completed+1} 步：({r+1}, {c+1}) → ({nr+1}, {nc+1})，立即消除 {move["score"]} 格，預估連鎖 {move["cascade_estimate"]} 次，下一手價值 {move["followup_value"]:.1f}，策略值 {move["priority"]:.1f}')
            pyg.moveTo(*gem_position(r, c), duration=0)
            try:
                pyg.dragTo(*gem_position(nr, nc), duration=0.12, button='left')
            finally:
                # 中斷時也釋放按鍵，不關閉 PyAutoGUI 的緊急停止。
                pyg.mouseUp(_pause=False)
            # 拖曳後停在目標格，下一步直接移到新起點，不再繞回固定位置。
            previous = [row[:] for row in board]
            last_move = (move['start'], move['end'])
            completed += 1
            # 留時間讓交換、掉落與連鎖動畫開始，再等穩定。
            pyg.sleep(min(0.25, max(0, deadline-time.monotonic())))
    except (pyg.FailSafeException, KeyboardInterrupt):
        print('已緊急停止。')
    except (TimeoutError, ValueError) as exc:
        print('已安全停止：', exc)
    finally:
        print(f'自動操作結束，共嘗試 {completed} 次交換。')

def main():
    try:
        countdown('5 秒後開始高速取色，請切到遊戲並等待棋盤靜止。', seconds=5)
        park_mouse()
        print('開始 85 秒高速循環：一次讀取 64 格 → 判斷 → 交換 → 等待穩定 → 重新讀取。')
        auto_play(seconds=85)
    except (pyg.FailSafeException, KeyboardInterrupt):
        print('已停止。')
    except (TimeoutError, ValueError) as exc:
        print('請修正後重新執行 main()：', exc)


In [ ]:
def test_logic():
    # 純模擬測試：不截圖、不移動滑鼠。
    board = [[f'cell-{r}-{c}' for c in range(SIZE)] for r in range(SIZE)]
    board[0][0] = board[0][2] = board[1][1] = '紅'
    original = [row[:] for row in board]
    assert can_move_up(1, 1, board)
    assert can_move_down(0, 1, board)
    assert len(swap_matches(board, (1, 1), (0, 1))) == 3
    assert not can_move_left(0, 0, board)
    assert not can_move_right(0, 7, board)
    assert not can_move_up(0, 0, board)
    assert not can_move_down(7, 0, board)
    assert not swap_matches(board, (0, 0), (2, 0))
    assert board == original
    transposed = [list(row) for row in zip(*board)]
    assert can_move_left(1, 1, transposed)
    assert can_move_right(1, 0, transposed)
    moves = find_moves(board)
    keys = [tuple(sorted((m['start'], m['end']))) for m in moves]
    assert len(keys) == len(set(keys))
    assert any(m['start'] == (0, 1) and m['end'] == (1, 1) for m in moves)
    unknown = [row[:] for row in board]
    unknown[1][1] = '?'
    assert not can_move_up(1, 1, unknown)
    print('四方向、邊界、未知色、去重及棋盤不變性測試通過。')

test_logic()


In [ ]:
def test_strategy():
    base = [[f'{r}-{c}' for c in range(SIZE)] for r in range(SIZE)]
    base[0][0] = base[0][1] = base[0][3] = base[1][2] = '紅'
    base[3][0] = base[3][2] = base[4][1] = '藍'
    original = [row[:] for row in base]
    options = [m for m in find_moves(base) if (m['start'], m['end']) in
               [((0, 2), (1, 2)), ((3, 1), (4, 1))]]
    assert len(options) == 2
    ranked = rank_moves(base, options, budget=0)
    assert ranked[0]['score'] == 4
    assert base == original
    assert all('_trial' not in m and '_matched' not in m for m in ranked)
    assert pattern_value([{(0, c) for c in range(5)}]) > pattern_value([{(0, c) for c in range(4)}])
    falling = [[f'x{r}-{c}' for c in range(SIZE)] for r in range(SIZE)]
    for c in range(3):
        falling[7][c], falling[6][c] = '紅', '綠'
    dropped = remove_and_drop(falling, {(7, c) for c in range(3)})
    assert all(dropped[0][c] is None for c in range(3))
    assert {(7, c) for c in range(3)} <= all_matches(dropped)
    assert not all_matches([[None]*SIZE for _ in range(SIZE)])
    assert rank_moves(base, [], budget=0) == []
    assert best_followup_value(base, time.monotonic()+1) > 0
    assert best_followup_value([[None]*SIZE for _ in range(SIZE)], time.monotonic()+1) == 0
    assert best_followup_value(base, time.monotonic()-1) == 0
    planning = rank_moves(base, options, budget=0.2, remaining=85)
    endgame = rank_moves(base, options, budget=0.2, remaining=2)
    assert all(m['followup_value'] == 0 for m in endgame)
    assert any(m['followup_value'] > 0 for m in planning)
    assert all(not any(k.startswith('_') for k in m) for m in planning)
    assert base == original
    partial = [row[:] for row in base]
    partial[7][7] = '?'
    assert find_moves(partial)
    assert all(m['followup_value'] == 0 for m in rank_moves(partial, find_moves(partial), budget=0.2))
    frames = []
    for frame_index in range(STABLE_FRAMES):
        colors = [row[:] for row in base]
        colors[7][7] = '紅' if frame_index % 2 else '藍'
        pixels = [[(100, 100, 100) for _ in range(SIZE)] for _ in range(SIZE)]
        frames.append((colors, pixels))
    stable, _ = stable_cells_from_frames(frames)
    assert stable[7][7] == '?' and stable[0][0] == base[0][0]
    assert frames[-1][0][7][7] != '?'
    assert stable_cells_from_frames(frames[:1]) is None
    for colors, pixels in frames:
        for c in range(MAX_UNCERTAIN_CELLS+1):
            colors[0][c] = '?'
    assert stable_cells_from_frames(frames) is None
    marked = GemBoard([row[:] for row in base], {(4, 1)})
    preferred = rank_moves(marked, find_moves(marked), budget=0)
    assert preferred[0]['black_hits'] == 1
    blue_move = next(m for m in preferred if m['start'] == (3, 1) and m['end'] == (4, 1))
    assert blue_move['black_hits'] == 1
    displaced = GemBoard([row[:] for row in base], {(3, 1)})
    alternative = rank_moves(displaced, find_moves(displaced), budget=0)
    assert next(m for m in alternative if m['start'] == (3, 1) and m['end'] == (4, 1))['black_hits'] == 0
    from PIL import Image, ImageDraw
    for color in ('yellow', 'red', 'blue', 'white'):
        image = Image.new('RGB', (50, 50), color)
        assert not has_inner_black(image, 25, 25)
        ImageDraw.Draw(image).rectangle((23, 23, 26, 26), fill='black')
        assert has_inner_black(image, 25, 25)
    assert not has_inner_black(Image.new('RGB', (50, 50), 'black'), 25, 25)
    print('策略測試通過：黑色標記優先、交換位置追蹤、不同底色偵測、暗背景排除及特殊格防卡住。')

test_strategy()


In [ ]:
# 執行後等 5 秒 → 一次截圖讀取 64 格 → 自動交換並重新取色，共 85 秒。
main()